In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/wangapa106g/afri-deep-tech-using-gemma3-270m-param/__results__.html
/kaggle/input/notebooks/wangapa106g/afri-deep-tech-using-gemma3-270m-param/__huggingface_repos__.json
/kaggle/input/notebooks/wangapa106g/afri-deep-tech-using-gemma3-270m-param/__notebook__.ipynb
/kaggle/input/notebooks/wangapa106g/afri-deep-tech-using-gemma3-270m-param/__output__.json
/kaggle/input/notebooks/wangapa106g/afri-deep-tech-using-gemma3-270m-param/custom.css
/kaggle/input/notebooks/wangapa106g/afri-deep-tech-using-gemma3-270m-param/reports/final_model_report.json
/kaggle/input/notebooks/wangapa106g/afri-deep-tech-using-gemma3-270m-param/llama.cpp/CMakePresets.json
/kaggle/input/notebooks/wangapa106g/afri-deep-tech-using-gemma3-270m-param/llama.cpp/CONTRIBUTING.md
/kaggle/input/notebooks/wangapa106g/afri-deep-tech-using-gemma3-270m-param/llama.cpp/mypy.ini
/kaggle/input/notebooks/wangapa106g/afri-deep-tech-using-gemma3-270m-param/llama.cpp/convert_lora_to_gguf.py
/kaggle/input/noteboo

In [2]:
# ============================================================
# CELL 1 — ADTC PROFILER + LLAMA.CPP SETUP
# ============================================================

import os
import sys
import subprocess

print("Python:", sys.version)

# ------------------------------------------------------------
# 1. Install ADTC profiler
# ------------------------------------------------------------
print("\n[1/3] Installing ADTC profiler...")

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "-q",
    "git+https://github.com/Africa-Deep-Tech-Foundation/adtc-profiler.git"
], check=True)

# ------------------------------------------------------------
# 2. Install llama.cpp tools
#
# We specifically need llama-bench.
# Build from source so the binary is available.
# ------------------------------------------------------------
print("\n[2/3] Installing/building llama.cpp...")

LLAMA_DIR = "/kaggle/working/llama.cpp"

if not os.path.exists(LLAMA_DIR):
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/ggml-org/llama.cpp.git",
        LLAMA_DIR
    ], check=True)

# Build llama.cpp
subprocess.run(
    ["cmake", "-B", "build", "-DGGML_NATIVE=ON"],
    cwd=LLAMA_DIR,
    check=True
)

subprocess.run(
    ["cmake", "--build", "build", "--config", "Release", "-j2"],
    cwd=LLAMA_DIR,
    check=True
)

# ------------------------------------------------------------
# 3. Add llama.cpp binaries to PATH
# ------------------------------------------------------------
BIN_DIRS = [
    os.path.join(LLAMA_DIR, "build", "bin"),
    os.path.join(LLAMA_DIR, "build", "bin", "Release"),
]

for d in BIN_DIRS:
    if os.path.isdir(d):
        os.environ["PATH"] = d + os.pathsep + os.environ["PATH"]

print("\n[3/3] Setup complete.")
print("llama.cpp:", LLAMA_DIR)
print("PATH updated.")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

[1/3] Installing ADTC profiler...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 MB 26.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 3.6 MB/s eta 0:00:00

[2/3] Installing/building llama.cpp...


Cloning into '/kaggle/working/llama.cpp'...


-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.1.0-dev
-- Found Git: /usr/bin/git (found version "2.34.1")


CMAKE_BUILD_TYPE=Release


-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found OpenMP_C: -fopenmp (found version "4.5")
-- Found OpenMP_CXX: -fopenmp (found version "4.5")
-- Found OpenMP: TRUE (found version "4.5")
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.20.1
-- ggml commit:  f9779dd
-- Found OpenSSL: /usr/lib/x86_64-linux-gnu/libcrypto.so (found version "3.0.2")
-- Performing Test OPENSSL_VERSION_SUPPORTED
-- Performing Test OPENSSL_VERSION_SUPPORTED - Success
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: llama-app
-- Configuring done (1.4s)
-- Generating done (0.4s)
-- Build files have

In [3]:
# ============================================================
# CELL 2 — VERIFY MODEL + LLAMA-BENCH + ADTC PROFILER
# ============================================================

import os
import shutil
import subprocess
from pathlib import Path

MODEL_PATH = Path(
    "/kaggle/input/notebooks/wangapa106g/"
    "afri-deep-tech-using-gemma3-270m-param/"
    "models/gguf/"
    "gemma3-financial-intelligence-Q4_K_M.gguf"
)

print("=" * 70)
print("ADTC MODEL / ENVIRONMENT VERIFICATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify model
# ------------------------------------------------------------
print("\n[MODEL]")

if MODEL_PATH.exists():
    size_gb = MODEL_PATH.stat().st_size / (1024 ** 3)

    print("✓ Model exists")
    print("  Path :", MODEL_PATH)
    print(f"  Size : {size_gb:.3f} GB")
else:
    print("✗ MODEL NOT FOUND")
    print("  Expected path:")
    print(" ", MODEL_PATH)

# ------------------------------------------------------------
# 2. Verify llama-bench
# ------------------------------------------------------------
print("\n[LLAMA-BENCH]")

llama_bench = shutil.which("llama-bench")

if llama_bench:
    print("✓ llama-bench found")
    print("  Path:", llama_bench)

    result = subprocess.run(
        [llama_bench, "--help"],
        capture_output=True,
        text=True
    )

    print("  Executable check: OK")
else:
    print("✗ llama-bench NOT FOUND")

# ------------------------------------------------------------
# 3. Verify llama-cli
# ------------------------------------------------------------
print("\n[LLAMA-CLI]")

llama_cli = shutil.which("llama-cli")

if llama_cli:
    print("✓ llama-cli found")
    print("  Path:", llama_cli)
else:
    print("⚠ llama-cli not found")

# ------------------------------------------------------------
# 4. Verify ADTC profiler
# ------------------------------------------------------------
print("\n[ADTC PROFILER]")

profiler = shutil.which("adtc-profiler")

if profiler:
    print("✓ adtc-profiler found")
    print("  Path:", profiler)

    result = subprocess.run(
        ["adtc-profiler", "--help"],
        capture_output=True,
        text=True
    )

    print("\nProfiler help:")
    print(result.stdout[:3000])
else:
    print("✗ adtc-profiler NOT FOUND")

# ------------------------------------------------------------
# 5. Verify llama-bench can actually see the GGUF
# ------------------------------------------------------------
if MODEL_PATH.exists() and llama_bench:

    print("\n[LLAMA-BENCH MODEL LOAD TEST]")
    print("Running a short benchmark/load test...")

    result = subprocess.run(
        [
            llama_bench,
            "-m", str(MODEL_PATH),
            "-p", "32",
            "-n", "32",
            "-r", "1"
        ],
        capture_output=True,
        text=True
    )

    print("\nSTDOUT:")
    print(result.stdout[-5000:])

    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr[-3000:])

    print("\nReturn code:", result.returncode)

    if result.returncode == 0:
        print("✓ llama-bench successfully loaded the GGUF")
    else:
        print("✗ llama-bench failed to load the model")

print("\n" + "=" * 70)
print("VERIFICATION COMPLETE")
print("=" * 70)

ADTC MODEL / ENVIRONMENT VERIFICATION

[MODEL]
✓ Model exists
  Path : /kaggle/input/notebooks/wangapa106g/afri-deep-tech-using-gemma3-270m-param/models/gguf/gemma3-financial-intelligence-Q4_K_M.gguf
  Size : 0.236 GB

[LLAMA-BENCH]
✓ llama-bench found
  Path: /kaggle/working/llama.cpp/build/bin/llama-bench
  Executable check: OK

[LLAMA-CLI]
✓ llama-cli found
  Path: /kaggle/working/llama.cpp/build/bin/llama-cli

[ADTC PROFILER]
✓ adtc-profiler found
  Path: /usr/local/bin/adtc-profiler

Profiler help:
Usage: adtc-profiler [OPTIONS] COMMAND [ARGS]...

  ADTC 2026 reference profiler.

Options:
  --help  Show this message and exit.

Commands:
  compare  Diff a participant submission.json against an audit.json.
  run      Run the full profiler pipeline and emit a schema-valid JSON...


[LLAMA-BENCH MODEL LOAD TEST]
Running a short benchmark/load test...

STDOUT:
| model                          |       size |     params | backend    | threads |            test |                  t/s |
| 

In [4]:
# ============================================================
# CELL 3A — INSPECT ADTC SUBMISSION FORMAT
# ============================================================

import os
import subprocess
from pathlib import Path

print("=" * 70)
print("INSPECTING ADTC PROFILER SUBMISSION FORMAT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Locate installed profiler package
# ------------------------------------------------------------

result = subprocess.run(
    ["python", "-c",
     "import adtc_profiler, os; print(os.path.dirname(adtc_profiler.__file__))"],
    capture_output=True,
    text=True
)

print("\nInstalled package:")
print(result.stdout)

PACKAGE_DIR = Path(result.stdout.strip())

# ------------------------------------------------------------
# 2. Search for metadata.json examples/templates
# ------------------------------------------------------------

print("\nSearching for metadata.json...")
print("-" * 70)

matches = []

for root, dirs, files in os.walk(PACKAGE_DIR):
    for file in files:
        if file == "metadata.json":
            matches.append(Path(root) / file)

if matches:
    for path in matches:
        print("\nFOUND:")
        print(path)
        print("\nCONTENT:")
        print(path.read_text()[:10000])
else:
    print("No metadata.json inside package.")

# ------------------------------------------------------------
# 3. Search profiler source for metadata requirements
# ------------------------------------------------------------

print("\nSearching source code for 'metadata.json'...")
print("-" * 70)

for root, dirs, files in os.walk(PACKAGE_DIR):
    for file in files:
        if file.endswith(".py"):
            path = Path(root) / file

            try:
                text = path.read_text(errors="ignore")
            except:
                continue

            if "metadata.json" in text:
                print("\nSOURCE:", path)

                lines = text.splitlines()

                for i, line in enumerate(lines):
                    if "metadata.json" in line:
                        start = max(0, i - 8)
                        end = min(len(lines), i + 15)

                        print("\n".join(
                            f"{j+1}: {lines[j]}"
                            for j in range(start, end)
                        ))

# ------------------------------------------------------------
# 4. Search installed package for examples/submission files
# ------------------------------------------------------------

print("\nSearching for submission-related files...")
print("-" * 70)

for root, dirs, files in os.walk(PACKAGE_DIR):
    for file in files:
        lower = file.lower()

        if (
            "submission" in lower
            or "metadata" in lower
            or "schema" in lower
        ):
            print(Path(root) / file)

print("\n" + "=" * 70)
print("INSPECTION COMPLETE")
print("=" * 70)

INSPECTING ADTC PROFILER SUBMISSION FORMAT

Installed package:
/usr/local/lib/python3.12/dist-packages/adtc_profiler


Searching for metadata.json...
----------------------------------------------------------------------
No metadata.json inside package.

Searching source code for 'metadata.json'...
----------------------------------------------------------------------

SOURCE: /usr/local/lib/python3.12/dist-packages/adtc_profiler/cli.py
4:     adtc-profiler run \\
5:         --submission path/to/submission-dir \\
6:         --mode {participant,audit} \\
7:         --output audit.json
8:         [--skip-accuracy]
9:         [--seed 42]
10: 
11: The submission directory must contain:
12:   - metadata.json   submission claims (team_id, domain, language_scope, ...)
13:   - model file referenced by metadata.json
14: """
15: from __future__ import annotations
16: 
17: import json
18: import sys
19: from pathlib import Path
20: 
21: import click
22: from rich.console import Console
23: 
24: f

In [5]:
# ============================================================
# INSPECT THE ACTUAL ADTC METADATA SCHEMA
# ============================================================

import json
from pathlib import Path

schema_path = Path(
    "/usr/local/lib/python3.12/dist-packages/"
    "adtc_profiler/schema/adtc-profiler.schema.json"
)

print("=" * 70)
print("ADTC PROFILER SCHEMA")
print("=" * 70)

schema = json.loads(schema_path.read_text())

print(json.dumps(schema, indent=2))

print("\n" + "=" * 70)
print("TOP-LEVEL PROPERTIES")
print("=" * 70)

for name, definition in schema.get("properties", {}).items():
    print(f"\n{name}:")
    print(json.dumps(definition, indent=2))

ADTC PROFILER SCHEMA
{
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "$id": "https://adtc.africa/schemas/adtc-profiler.schema.json",
  "title": "ADTC Profiler Submission",
  "type": "object",
  "additionalProperties": false,
  "required": [
    "schema_version",
    "profiler_version",
    "submission",
    "environment",
    "throughput",
    "memory",
    "accuracy",
    "cpu_thermal",
    "reproducibility"
  ],
  "properties": {
    "schema_version": {
      "type": "string",
      "pattern": "^[0-9]+\\.[0-9]+\\.[0-9]+$"
    },
    "profiler_version": {
      "type": "string",
      "minLength": 1
    },
    "submission": {
      "type": "object",
      "additionalProperties": false,
      "required": [
        "team_id",
        "domain",
        "language_scope",
        "african_alpha_claim",
        "budget_laptop_claim",
        "submitter",
        "cross_disciplinary_pairing",
        "test_prompts",
        "model"
      ],
      "properties": {
        "tea

In [6]:
# ============================================================
# CELL 3 — PREPARE + RUN ADTC PROFILER
# ============================================================

import json
import shutil
import subprocess
from pathlib import Path

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

SOURCE_MODEL = Path(
    "/kaggle/input/notebooks/wangapa106g/"
    "afri-deep-tech-using-gemma3-270m-param/"
    "models/gguf/"
    "gemma3-financial-intelligence-Q4_K_M.gguf"
)

SUBMISSION_DIR = Path("/kaggle/working/adtc_submission")
MODEL_DEST = SUBMISSION_DIR / "model.gguf"
METADATA_PATH = SUBMISSION_DIR / "metadata.json"
OUTPUT_PATH = Path("/kaggle/working/submission.json")

# ------------------------------------------------------------
# ADTC Submission Metadata
# ------------------------------------------------------------

metadata = {
    "team_id": "REPLACE_WITH_YOUR_TEAM_ID",

    "domain": "corporate_enterprise",

    "language_scope": [
        "en"
    ],

    "african_alpha_claim": True,

    "budget_laptop_claim": True,

    "submitter": {
        "name": "REPLACE_WITH_YOUR_NAME",
        "email": "REPLACE_WITH_YOUR_EMAIL",
        "github_handle": "REPLACE_WITH_YOUR_GITHUB"
    },

    "cross_disciplinary_pairing": {
        "discipline": "Financial intelligence",
        "load_bearing": True,
        "description": (
            "Financial data analysis and SME financial intelligence "
            "are central to the model's intended use."
        )
    },

    # --------------------------------------------------------
    # ACTUAL MODEL TASK:
    # M-Pesa SMS -> 5-field structured JSON
    # --------------------------------------------------------

    "test_prompts": [
        {
            "prompt_id": "mpesa_extraction_income",
            "prompt": """Extract the M-Pesa transaction from the SMS below and return ONLY valid JSON.

The JSON MUST contain exactly these five fields:
- entity
- amount
- balance
- date
- type

Rules:
- amount must be a number.
- balance must be a number.
- date must use YYYY-MM-DD format.
- type must be either "income" or "expense".
- entity must contain the relevant person, business, or organization.
- Do not add any extra fields.
- Do not include markdown, explanations, or code fences.

Transaction date: 2026-03-05

SMS:
You have received Ksh20,000.00 from Ann Mueni. Transaction cost Ksh0.00. New balance Ksh159,583.00.

Return ONLY the JSON object."""
        },
        {
            "prompt_id": "mpesa_extraction_expense",
            "prompt": """Extract the M-Pesa transaction from the SMS below and return ONLY valid JSON.

The JSON MUST contain exactly these five fields:
- entity
- amount
- balance
- date
- type

Rules:
- amount must be a number.
- balance must be a number.
- date must use YYYY-MM-DD format.
- type must be either "income" or "expense".
- entity must contain the relevant person, business, or organization.
- Do not add any extra fields.
- Do not include markdown, explanations, or code fences.

Transaction date: 2026-03-06

SMS:
You paid Ksh1,500.00 to Naivas Supermarket. Transaction cost Ksh0.00. New M-PESA balance is Ksh158,083.00.

Return ONLY the JSON object."""
        }
    ],

    # --------------------------------------------------------
    # Model information
    # --------------------------------------------------------

    "model": {
        "name": "Gemma 3 270M Financial Intelligence",
        "runtime": "llama.cpp",
        "quantization": "Q4_K_M",
        "parameters_estimate": "270M",
        "packaging": "binary_bundle"
    },

    # --------------------------------------------------------
    # Runtime path
    # --------------------------------------------------------

    "_runtime": {
        "model_path": "model.gguf"
    }
}

# ------------------------------------------------------------
# Recreate submission directory
# ------------------------------------------------------------

if SUBMISSION_DIR.exists():
    shutil.rmtree(SUBMISSION_DIR)

SUBMISSION_DIR.mkdir(parents=True)

# ------------------------------------------------------------
# Verify source model
# ------------------------------------------------------------

if not SOURCE_MODEL.exists():
    raise FileNotFoundError(
        f"Fine-tuned GGUF model not found:\n{SOURCE_MODEL}"
    )

print("=" * 70)
print("PREPARING ADTC SUBMISSION")
print("=" * 70)

# ------------------------------------------------------------
# Copy fine-tuned GGUF
# ------------------------------------------------------------

print("\nCopying fine-tuned model...")
print("FROM :", SOURCE_MODEL)
print("TO   :", MODEL_DEST)

shutil.copy2(SOURCE_MODEL, MODEL_DEST)

print("✓ Model copied")

# ------------------------------------------------------------
# Write metadata.json
# ------------------------------------------------------------

METADATA_PATH.write_text(
    json.dumps(metadata, indent=2)
)

print("✓ metadata.json created")

# ------------------------------------------------------------
# Verify submission structure
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SUBMISSION STRUCTURE")
print("=" * 70)

for path in SUBMISSION_DIR.iterdir():
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"{path.name:20s} {size_mb:.2f} MB")

# ------------------------------------------------------------
# Display metadata
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("METADATA")
print("=" * 70)

print(METADATA_PATH.read_text())

# ------------------------------------------------------------
# Run FULL ADTC participant profiler
#
# Accuracy is intentionally enabled.
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RUNNING FULL ADTC PARTICIPANT PROFILER")
print("=" * 70)

cmd = [
    "adtc-profiler",
    "run",
    "--submission", str(SUBMISSION_DIR),
    "--mode", "participant",
    "--output", str(OUTPUT_PATH)
]

print("\n$ " + " ".join(cmd))
print()

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("STDOUT")
print("-" * 70)
print(result.stdout)

if result.stderr:
    print("\nSTDERR")
    print("-" * 70)
    print(result.stderr)

print("\nReturn code:", result.returncode)

# ------------------------------------------------------------
# Display final report
# ------------------------------------------------------------

if OUTPUT_PATH.exists():

    print("\n" + "=" * 70)
    print("✓ FULL ADTC REPORT GENERATED")
    print("=" * 70)

    report = json.loads(
        OUTPUT_PATH.read_text()
    )

    print(
        json.dumps(
            report,
            indent=2
        )
    )

else:
    print("\n✗ submission.json was not generated.")

PREPARING ADTC SUBMISSION

Copying fine-tuned model...
FROM : /kaggle/input/notebooks/wangapa106g/afri-deep-tech-using-gemma3-270m-param/models/gguf/gemma3-financial-intelligence-Q4_K_M.gguf
TO   : /kaggle/working/adtc_submission/model.gguf
✓ Model copied
✓ metadata.json created

SUBMISSION STRUCTURE
model.gguf           241.39 MB
metadata.json        0.00 MB

METADATA
{
  "team_id": "REPLACE_WITH_YOUR_TEAM_ID",
  "domain": "corporate_enterprise",
  "language_scope": [
    "en"
  ],
  "african_alpha_claim": true,
  "budget_laptop_claim": true,
  "submitter": {
    "name": "REPLACE_WITH_YOUR_NAME",
    "email": "REPLACE_WITH_YOUR_EMAIL",
    "github_handle": "REPLACE_WITH_YOUR_GITHUB"
  },
  "cross_disciplinary_pairing": {
    "discipline": "Financial intelligence",
    "load_bearing": true,
    "description": "Financial data analysis and SME financial intelligence are central to the model's intended use."
  },
  "test_prompts": [
    {
      "prompt_id": "mpesa_extraction_income",
    